In [ ]:
import time
import RNA
from nupack import *
import random
import math

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in DNA)

def generate_random_dna(length=20):
    bases = ['A', 'T', 'C', 'G']
    return ''.join(random.choices(bases, k=length))

def GC_content(seq: str) -> float:
    seq = seq.upper()
    valid_bases = [base for base in seq if base in {'A', 'C', 'G', 'T'}]
    if not valid_bases:
        return 0.0
    gc_count = sum(1 for base in valid_bases if base in {'G', 'C'})
    return gc_count / len(valid_bases)


def has_homopolymer(seq: str, run_length: int = 3) -> bool:
    if run_length < 2:
        raise ValueError("run_length must be at least 2")

    seq = seq.upper()
    count = 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            count += 1
            if count >= run_length:
                return True
        else:
            count = 1
    return False

def find_parentheses_pairs(s):
    stack = []  # Stack to keep track of '(' positions
    pairs = {}  # Dictionary to store pairs of indices (key is '(' or ')', value is the paired parenthesis index)
    
    # Iterate through the string with index
    for i, char in enumerate(s):
        if char == '(':
            stack.append(i)  # Push the index of '(' onto the stack
        elif char == ')':
            if stack:  # Ensure stack is not empty
                open_index = stack.pop()  # Pop the index of the matching '('
                pairs[open_index] = i  # Store the pair (open_index, close_index)
                pairs[i] = open_index  # Store the reverse pair (close_index, open_index)
    
    return pairs

def has_consecutive_parens(s, num) -> bool:
    return '(' * num in s and ')' * num in s


start_time = time.time()             # Record start time
duration = 1200                     

crRNA_scaffold = 'UAAUUUCUACUCUUGUAGAU'  # Direct repeat sequence for AsCas12a
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

i = 0

with open('generated_random_dna_6_pairs_seed_complete_paired_end_complete_unpaired.txt', 'w') as outfile:
    while (time.time() - start_time) < duration:
        random_dna = generate_random_dna()
        spacer = DNA_to_RNA(random_dna)
        crRNA = crRNA_scaffold + spacer

        # Compute partition function and suboptimal structures
        partition_function_guide = pfunc(strands=crRNA, model=my_model_RNA)
        ensemble_energy_guide = partition_function_guide[1]

        subopt_structures_guide = subopt(strands=crRNA, energy_gap=1, model=my_model_RNA)

        kT = RNA.exp_param().kT / 1000.0
        prob_sub_guide = math.exp((ensemble_energy_guide - subopt_structures_guide[0].energy) / kT)
        struct = str(subopt_structures_guide[0].structure)

        # Check structure and probability threshold
        if GC_content(random_dna) >= 0.4 and GC_content(random_dna) <= 0.6 and has_homopolymer(random_dna) == False:
            if struct[0:26] == '.....(((((....))))).((((((' and ')' not in struct[39:40]:
                pairs_num = len(find_parentheses_pairs(struct[20:])) / 2
                if pairs_num == 6 and prob_sub_guide >= 0.65 and has_consecutive_parens(struct[20:], 6) == True:
                    prob_matrix = pairs(strands=crRNA, model=my_model_RNA).to_array()
                    flag = 1
                    for k in list(find_parentheses_pairs(struct[20:])):
                        if prob_matrix[k+20][find_parentheses_pairs(struct[20:])[k]+20] < 0.7:
                            flag = -1
                            break
                    
                    if flag == 1:
                        # Print to console (optional)
                        print(i+1)
                        print(random_dna)
                        print(crRNA)
                        print(struct)
                        print(prob_sub_guide)
                        print()
                  
                        # Write just the random_dna to file (one per line)
                        outfile.write(random_dna + '\n')
                        i += 1
